# BOS Flight Data Clean

Target：Estimate delay for more than 15 minutes（ArrDel15），and cancellation（Cancelled）。
Train set: 2022—2025
Test set: 2026 Jan — Jul


In [1]:
from pathlib import Path
import zipfile
import pandas as pd

#Data address
project_folder = Path(r"C:\Users\mengk\OneDrive\文档\大四\CS 506\Final Project")

flight_folder = project_folder / "data" / "raw data" / "bts"
weather_file = project_folder / "data" / "raw data" / "iem" / "iem_BOS_2022_2026_07.csv"
save_folder = project_folder / "data" / "processed_basic"


low_visibility_limit = 1.0 # low visibility limit in miles
weather_time_limit = 120  # weather data time limit in minutes before departure

## 1. read 55 months flight data，keep data Origin == "BOS"


In [13]:
flight_columns = [
    "Year", "Month", "DayOfWeek", "FlightDate",
    "Reporting_Airline", "Flight_Number_Reporting_Airline",
    "Origin", "Dest", "CRSDepTime", "CRSArrTime",
    "Distance", "ArrDelay", "ArrDel15", "Cancelled", "Diverted"
]

monthly_tables = []
month_counts = []


for year in range(2022, 2027):
    if year == 2026:
        last_month = 7
    else:
        last_month = 12

    # Read the monthly flight data
    for month in range(1, last_month + 1):
        zip_file = flight_folder / f"bts_{year}_{month}.zip"
        csv_name = f"On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_{year}_{month}.csv"

        with zipfile.ZipFile(zip_file) as zip:
            with zip.open(csv_name) as csv_file:
                # read csv and record cols in flight_columns; regard as string for now
                month_flights = pd.read_csv(csv_file, usecols=flight_columns, dtype=str)

        # remove space and covert to upper case
        month_flights["Origin"] = month_flights["Origin"].str.strip().str.upper()

        # Keep if origin is BOS
        bos = month_flights[month_flights["Origin"] == "BOS"].copy()


        # Format string data into data frame date
        dates = pd.to_datetime(bos["FlightDate"], format="%Y-%m-%d")

        # Check if there is problem with date
        correct_date = (dates.dt.year == year) & (dates.dt.month == month)
        date_problems = len(bos) - correct_date.sum()

        # Merge data
        monthly_tables.append(bos)
        month_counts.append([year, month, len(bos), date_problems])

        print(year, month, "Number of Flights:", len(bos), "Date Problems:", date_problems, flush=True)

# Merge into one DataFrame
flights = pd.concat(monthly_tables, ignore_index=True)
# Create a DataFrame to store coverage information
coverage = pd.DataFrame(month_counts, columns=["Year", "Month", "BOSRows", "DateProblems"])

print("Number of Months:", len(coverage))
print("Total BOS Records:", len(flights))

del monthly_tables, month_flights, bos

2022 1 Number of Flights: 9917 Date Problems: 0
2022 2 Number of Flights: 9488 Date Problems: 0
2022 3 Number of Flights: 11073 Date Problems: 0
2022 4 Number of Flights: 11430 Date Problems: 0
2022 5 Number of Flights: 12116 Date Problems: 0
2022 6 Number of Flights: 11688 Date Problems: 0
2022 7 Number of Flights: 12179 Date Problems: 0


KeyboardInterrupt: 

## 2. Clean up text, and convert dates and numerical values

Use `strip` to remove leading and trailing whitespace, `upper` to standardize text to uppercase, convert to numeric format, and `replace` to treat empty strings as missing values.

In [3]:
# Clean and format the missing values
for column in flight_columns:
    flights[column] = flights[column].str.strip()
flights = flights.replace("", pd.NA)
flights = flights.replace("M", pd.NA)

for column in ["Reporting_Airline", "Origin", "Dest"]:
    flights[column] = flights[column].str.upper()

# Check for duplicate rows based on selected columns
flight_duplicate_count = flights.duplicated().sum()
print("Duplicate rows:", flight_duplicate_count)

# Convert data types for flightdate column
flights["FlightDate"] = pd.to_datetime(flights["FlightDate"], format="%Y-%m-%d")

for column in ["Distance", "ArrDelay", "ArrDel15", "Cancelled", "Diverted"]:

    # Numerical columns should be converted to numeric type, with errors coerced to NaN
    flights[column] = pd.to_numeric(flights[column], errors="coerce")

    # replace anomalous values 
    flights[column] = flights[column].replace([float("inf"), float("-inf")], float("nan"))

for column in ["ArrDel15", "Cancelled", "Diverted"]:
    # Check if the labels are only 0 or 1
    present_labels = flights[column].dropna()
    normal_count = present_labels.isin([0, 1]).sum()
    print(column, "Wrong labels: ", len(present_labels) - normal_count)

# Ensure that Distance is positive
flights.loc[flights["Distance"] <= 0, "Distance"] = float("nan")

# Check if ArrDel15 is consistent with ArrDelay
has_both = flights["ArrDelay"].notna() & flights["ArrDel15"].notna()
label_check = flights[has_both].copy()


expected_label = (label_check["ArrDelay"] >= 15).astype(int)
wrong_labels = (label_check["ArrDel15"] != expected_label).sum()

print("Inconsistent labels: ", wrong_labels)
del label_check
# count the number of missing values in each column
print(flights[flight_columns].isna().sum())

Duplicate rows: 0
ArrDel15 Wrong labels:  0
Cancelled Wrong labels:  0
Diverted Wrong labels:  0
Inconsistent labels:  0
Year                                   0
Month                                  0
DayOfWeek                              0
FlightDate                             0
Reporting_Airline                      0
Flight_Number_Reporting_Airline        0
Origin                                 0
Dest                                   0
CRSDepTime                             0
CRSArrTime                             0
Distance                               0
ArrDelay                           16325
ArrDel15                           16325
Cancelled                              0
Diverted                               0
dtype: int64


## 3. Convert scheduled time into a date-time value.

This process accounts for transitions related to daylight saving time (DST).


In [5]:
for column in ["CRSDepTime", "CRSArrTime"]:
    clock = pd.to_numeric(flights[column])
    hour = clock // 100  # 1441 // 100 = 14。
    minute = clock % 100  # 1441 % 100 = 41。

    normal_time = (hour >= 0) & (hour <= 23) & (minute >= 0) & (minute <= 59)
    valid_time = normal_time | (clock == 2400)

    print(column, "missing values", clock.isna().sum())
    print(column, "non-integer ", (clock % 1 != 0).sum())
    print(column, "invalid hour/minute values", len(clock) - valid_time.sum())

    if column == "CRSDepTime":
        departure_minutes = hour * 60 + minute
    else:
        flights["ArrHour"] = (hour % 24).astype(int)

# generate YYYY/MM/DD HH:MM datetime for departure
time_since_midnight = pd.to_timedelta(departure_minutes, unit="m")
flights["DepartureLocal"] = flights["FlightDate"] + time_since_midnight

flights["Year"] = flights["FlightDate"].dt.year
flights["Month"] = flights["DepartureLocal"].dt.month
flights["DayOfWeek"] = flights["DepartureLocal"].dt.day_of_week + 1  
flights["DepHour"] = flights["DepartureLocal"].dt.hour
flights["DepMinute"] = flights["DepartureLocal"].dt.minute

# Timezone conversion
flights["DepartureUTC"] = flights["DepartureLocal"].dt.tz_localize(
    "America/New_York", ambiguous="NaT", nonexistent="NaT"
)
flights["DepartureUTC"] = flights["DepartureUTC"].dt.tz_convert("UTC")
flights["DepartureUTC"] = flights["DepartureUTC"].astype("datetime64[ns, UTC]")

print("Anomoly Flights:", flights["DepartureUTC"].isna().sum())

CRSDepTime missing values 0
CRSDepTime non-integer  0
CRSDepTime invalid hour/minute values 0
CRSArrTime missing values 0
CRSArrTime non-integer  0
CRSArrTime invalid hour/minute values 0
Anomoly Flights: 0


## 4. Separate the tasks for "arrival delays" and "cancellations."

For the arrival delay task, apply the following filters: exclude cancelled flights, exclude diverted flights, and ensure the label is either 0 or 1.

For the cancellation task, only the validity of the cancellation label is required; do not remove flights simply because they lack an arrival label.

In [ ]:
delay_flights = flights[flights["Cancelled"] == 0].copy()
delay_flights = delay_flights[delay_flights["Diverted"] == 0].copy()
delay_flights = delay_flights[delay_flights["ArrDel15"] == 1].copy()
cancel_flights = flights[flights["Cancelled"] == 1].copy()

print("Flights: ", len(flights))
print("Delay Flights: ", len(delay_flights))
print("Cancel Flights: ", len(cancel_flights))

## 5. Read and clean weather data

tmpf = temperature (°F),   
relh = relative humidity (%),  
sknt = wind speed (knots),  
gust = wind gust (knots), 
vsby = visibility (miles),  
skyc1 = cloud category for the first layer.  
Missing gust values ​​are left as-is; they are not replaced with 0.

In [7]:
weather_columns = ["station", "valid", "tmpf", "relh", "sknt", "gust", "vsby", "skyc1"]
weather = pd.read_csv(weather_file, usecols=weather_columns, dtype=str, na_values=["M"])

weather_raw_count = len(weather)

# Clean and format the missing values
for column in weather_columns:
    weather[column] = weather[column].str.strip()
weather = weather.replace("", pd.NA)
weather["station"] = weather["station"].str.upper()
weather["skyc1"] = weather["skyc1"].str.upper()

bos_weather_count = (weather["station"] == "BOS").sum()
print("Check anomoly station", len(weather) - bos_weather_count)

# Clean and format weather date data
weather["valid"] = pd.to_datetime(weather["valid"], format="%Y-%m-%d %H:%M", utc=True)
print("Check anomoly weather time", weather["valid"].isna().sum())
weather["valid"] = weather["valid"].astype("datetime64[ns, UTC]")

# Clean and format the numeric columns
for column in ["tmpf", "relh", "sknt", "gust", "vsby"]:
    weather[column] = pd.to_numeric(weather[column], errors="coerce")
    weather[column] = weather[column].replace([float("inf"), float("-inf")], float("nan"))

# Clean anomalous values for weather
weather.loc[weather["relh"] < 0, "relh"] = float("nan")
weather.loc[weather["relh"] > 100, "relh"] = float("nan")
for column in ["sknt", "gust", "vsby"]:
    weather.loc[weather[column] < 0, column] = float("nan")

cloud_types = ["CLR", "FEW", "SCT", "BKN", "OVC", "VV"]
known_clouds = weather["skyc1"].dropna()
normal_cloud_count = known_clouds.isin(cloud_types).sum()
print("Check anomoly cloud types", len(known_clouds) - normal_cloud_count)

# Create binary columns for low visibility and cloud types
weather["LowVisibility"] = 0.0
weather.loc[weather["vsby"] < low_visibility_limit, "LowVisibility"] = 1.0
weather.loc[weather["vsby"].isna(), "LowVisibility"] = float("nan")

for cloud in cloud_types:
    column = "Cloud_" + cloud
    weather[column] = 0.0
    weather.loc[weather["skyc1"] == cloud, column] = 1.0
    weather.loc[weather["skyc1"].isna(), column] = float("nan")

Check anomoly station 0
Check anomoly weather time 0
Check anomoly cloud types 0


## 6. Handling Duplicate Weather Observations for the Same Timestamp

Duplicate records exist for the same timestamp. An examination reveals that for each set of duplicates, one record is relatively complete, while the others contain significant amounts of missing data.

To resolve this, first sort the records by "fewer missing values ​​(priority)" and then by "higher source row number (priority)." Finally, remove duplicates based on the station and timestamp.


In [ ]:
weather_values = ["tmpf", "relh", "sknt", "gust", "vsby", "skyc1"]

weather["WeatherSourceRow"] = range(1, len(weather) + 1)
weather_duplicate_count = weather.duplicated(subset=["station", "valid"]).sum()

# sort_values in order to sort by missing count (ascending) and original row number (descending), keeping the latest record for ties.
weather["MissingCount"] = weather[weather_values].isna().sum(axis=1)
weather = weather.sort_values(["MissingCount", "WeatherSourceRow"], ascending=[True, False])
weather = weather.drop_duplicates(subset=["station", "valid"], keep="first")

weather = weather.drop(columns=["MissingCount"])
weather = weather.sort_values("valid")
weather = weather.reset_index(drop=True)

print("Raw records:", weather_raw_count)
print("Duplicate records:", weather_duplicate_count)
print("Final records:", len(weather))
print("Missing values:")
print(weather[weather_values].isna().sum())


# Gnerate local time and filter by date range
weather["LocalTime"] = weather["valid"].dt.tz_convert("America/New_York")
weather = weather[weather["LocalTime"].dt.tz_localize(None) >= "2022-01-01"].copy()
weather = weather[weather["LocalTime"].dt.tz_localize(None) < "2026-08-01"].copy()
weather["Year"] = weather["LocalTime"].dt.year
weather["Month"] = weather["LocalTime"].dt.month
weather["Hour"] = weather["LocalTime"].dt.hour



处理前： 40057
同站同时间额外记录： 14
处理后： 40043
tmpf         0
relh         2
sknt        12
gust     32112
vsby         4
skyc1        0
dtype: int64


## 7. Matching actual weather conditions with flights for correlation analysis

In [9]:
analysis_flights = flights[flights["Year"] <= 2025].copy()

weather_for_join = weather[
    [
        "valid", "WeatherSourceRow",
        "tmpf", "relh", "sknt", "gust", "vsby", "skyc1",
        "LowVisibility"
    ]
].copy()

# Rename
weather_for_join.columns = [
    "WeatherUTC", "WeatherSourceRow",
    "ActualTemp", "ActualHumidity", "ActualWind",
    "ActualGust", "ActualVisibility", "ActualCloud",
    "ActualLowVisibility"
]

# 3. Sort by time
analysis_flights = analysis_flights.sort_values("DepartureUTC")
weather_for_join = weather_for_join.sort_values("WeatherUTC")

# Find cloest historical weather data for each flight before flight using merge_asof
actual_weather = pd.merge_asof(
    analysis_flights,
    weather_for_join,
    left_on="DepartureUTC",
    right_on="WeatherUTC",
    direction="backward",
    tolerance=pd.Timedelta(minutes=weather_time_limit)
)

# 4. Calculate the time difference between the flight departure and the weather observation
time_difference = (
    actual_weather["DepartureUTC"] - actual_weather["WeatherUTC"]
)
actual_weather["WeatherAgeMinutes"] = time_difference.dt.total_seconds() / 60

print("Fail merge", actual_weather["WeatherUTC"].isna().sum())

Fail merge 72


## 8. Save Data

In [14]:
save_folder.mkdir(parents=True, exist_ok=True)


flights.to_csv(save_folder / "flights_clean.csv", index=False)

weather.to_csv(save_folder / "weather_clean.csv", index=False)

actual_weather.to_csv(
    save_folder / "actual_weather_2022_2025.csv",
    index=False
)

delay_flights.to_csv(
    save_folder / "delay_flights_clean.csv",
    index=False
)

cancel_flights.to_csv(
    save_folder / "cancel_flights_clean.csv",
    index=False
)

# coverage.to_csv(save_folder / "monthly_coverage.csv", index=False)


# save the summary report
report_names = [
    "Months",
    "AllFlights",
    "FlightDuplicatesBeforeAddingID",
    "DelayValidFlights",
    "CancellationValidFlights",
    "CancelledFlights",
    "DivertedFlights",
    "MissingArrivalLabel",
    "WeatherRaw",
    "WeatherClean",
    "WeatherDuplicateTimesBeforeCleaning",
    "WeatherDuplicateTimesAfterCleaning",
    "ActualWeatherFlights",
    "ActualWeatherUnmatched"
]

report_numbers = [
    len(coverage),
    len(flights),
    flight_duplicate_count,
    len(delay_flights),
    len(cancel_flights),
    (flights["Cancelled"] == 1).sum(),
    (flights["Diverted"] == 1).sum(),
    flights["ArrDel15"].isna().sum(),
    weather_raw_count,
    len(weather),
    weather_duplicate_count,
    weather.duplicated(subset=["station", "valid"]).sum(),
    len(actual_weather),
    actual_weather["WeatherUTC"].isna().sum()
]
report = pd.DataFrame({
    "Item": report_names,
    "Count": report_numbers
})

report.to_csv(save_folder / "quality_summary.csv", index=False)


# Save the missing value counts for each column in flights and weather
flight_missing = flights[flight_columns].isna().sum()
weather_missing = weather[weather_columns].isna().sum()

flight_missing.to_csv(
    save_folder / "flight_missing.csv",
    index_label="Column",
    header=["MissingCount"]
)

weather_missing.to_csv(
    save_folder / "weather_missing.csv",
    index_label="Column",
    header=["MissingCount"]
)



print(report)
print("Data saved to:", save_folder)

                                   Item   Count
0                                Months      55
1                            AllFlights  647451
2        FlightDuplicatesBeforeAddingID       0
3                     DelayValidFlights  142664
4              CancellationValidFlights   14576
5                      CancelledFlights   14576
6                       DivertedFlights    1749
7                   MissingArrivalLabel   16325
8                            WeatherRaw   40057
9                          WeatherClean   40043
10  WeatherDuplicateTimesBeforeCleaning      14
11   WeatherDuplicateTimesAfterCleaning       0
12                 ActualWeatherFlights  563342
13               ActualWeatherUnmatched      72
Data saved to: C:\Users\mengk\OneDrive\文档\大四\CS 506\Final Project\data\processed_basic
